In [ ]:
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    from sklearn.metrics import root_mean_squared_error
    _HAS_RMSE = True
except Exception:
    _HAS_RMSE = False


Xy_train = Xy_train
Xy_pool = Xy_aug
TEST_N = 299
SEEDS = list(range(10))
MODEL_SEED = 42


X_train_orig = Xy_train.drop(columns=[target_col])
y_train_orig = Xy_train[target_col]

print("Original train:", X_train_orig.shape, y_train_orig.shape)
print("Synthetic pool:", Xy_pool.shape)
print("Fixed TEST_N:", TEST_N, "| seeds:", len(SEEDS))


numeric_cols_orig = X_train_orig.select_dtypes(include=["int64", "float64"]).columns
categorical_cols_orig = X_train_orig.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor_orig = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols_orig),
        ("cat", categorical_transformer, categorical_cols_orig),
    ]
)


svr_model = SVR(kernel="rbf", C=100.0, epsilon=0.1, gamma="scale")
svr_pipe_orig = Pipeline(steps=[
    ("preprocessor", preprocessor_orig),
    ("model", svr_model),
])

svr_pipe_orig.fit(X_train_orig, y_train_orig)


rows = []

for sd in SEEDS:
    Xy_test = Xy_pool.sample(
        n=min(TEST_N, len(Xy_pool)),
        replace=False,
        random_state=sd
    )

    X_test_synth = Xy_test.drop(columns=[target_col])
    y_test_synth = Xy_test[target_col]

    y_pred = svr_pipe_orig.predict(X_test_synth)

    mse = mean_squared_error(y_test_synth, y_pred)
    rmse = root_mean_squared_error(y_test_synth, y_pred) if _HAS_RMSE else (mse ** 0.5)
    mae = mean_absolute_error(y_test_synth, y_pred)
    r2 = r2_score(y_test_synth, y_pred)

    rows.append({
        "seed": sd,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

df_res = pd.DataFrame(rows)

print("\n[Results per seed]")
print(df_res.round(4))

print("\n[Mean ± Std]")
summ = df_res[["MSE", "RMSE", "MAE", "R2"]].agg(["mean", "std"]).round(4)
print(summ)